# E1

Preparar los datos de pinguinos y comparar dos estrategias para predecir la especie:

- prediccion tradicional directa;
- prediccion jerarquica con dos modelos.

## Imports y configuracion

In [1]:
import numpy as np
import pandas as pd

# preparacion
from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

#modelos
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


## Carga de datos

In [2]:
df_original = pd.read_csv("data/E1_datos.csv")
df_original.head()

,Unnamed: 0,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [3]:
df_original.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         344 non-null    int64  
 1   species            344 non-null    str    
 2   island             344 non-null    str    
 3   culmen_length_mm   342 non-null    float64
 4   culmen_depth_mm    342 non-null    float64
 5   flipper_length_mm  308 non-null    float64
 6   body_mass_g        342 non-null    float64
 7   sex                334 non-null    str    
dtypes: float64(4), int64(1), str(3)
memory usage: 21.6 KB


## Mision 0: completando informacion

Se elimina la columna de indice que viene desde el CSV. Luego se imputan las columnas numericas con su promedio y se descartan las filas con informacion categorica faltante.

En `sex` aparece un valor `"."`, que no corresponde a una categoria valida de sexo. Lo trataremos como dato categorico faltante antes de descartar filas.

In [4]:
df = df_original.copy()
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

df["sex"] = df["sex"].replace(".", np.nan)

numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

missing_before = df.isna().sum()

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

rows_before_drop = len(df)
df = df.dropna(subset=categorical_cols).copy()
rows_after_drop = len(df)

summary_cleaning = pd.DataFrame({"faltantes_antes": missing_before, "faltantes_despues": df.isna().sum()})

print(f"filas originales: {len(df_original)}")
print(f"filas antes de descartar categoricas faltantes: {rows_before_drop}")
print(f"filas despues de descartar categoricas faltantes: {rows_after_drop}")
display(summary_cleaning)

filas originales: 344
filas antes de descartar categoricas faltantes: 344
filas despues de descartar categoricas faltantes: 333


,faltantes_antes,faltantes_despues
species,0,0
island,0,0
culmen_length_mm,2,0
culmen_depth_mm,2,0
flipper_length_mm,36,0
body_mass_g,2,0
sex,11,0


In [5]:
display(df.head())
display(df["species"].value_counts().rename("conteo_por_especie"))

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE


species
Adelie       146
Gentoo       119
Chinstrap     68
Name: conteo_por_especie, dtype: int64

## Mision 1: preparacion de los datos

Se separa la etiqueta `species` de las variables predictoras. Las variables categoricas `island`, `sex` se codifican con `OneHotEncoder` y las numericas se escalan con `StandardScaler`.

Para evitar data leakage, el codificado y el escalado se hace solo con el conjunto de entrenamiento mediante un `Pipeline`.

In [6]:
target = "species"
X = df.drop(columns=[target])
y = df[target]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Clases: {list(label_encoder.classes_)}")

Clases: ['Adelie', 'Chinstrap', 'Gentoo']


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Shape X_train:", X_train.shape)
print("Shape X_test:", X_test.shape)

Shape X_train: (266, 6)
Shape X_test: (67, 6)


In [8]:
categorical_features = ["island", "sex"]
numeric_features = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]

def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
            ("numericas", StandardScaler(), numeric_features),
        ]
    )

preprocessor_demo = make_preprocessor()
X_train_prepared = preprocessor_demo.fit_transform(X_train)
X_test_prepared = preprocessor_demo.transform(X_test)

print(f"columnas preparadas: {list(preprocessor_demo.get_feature_names_out())}")
print(f"shape X_train preparado: {X_train_prepared.shape}")
print(f"shape X_test preparado: {X_test_prepared.shape}")

columnas preparadas: ['categoricas__island_Biscoe', 'categoricas__island_Dream', 'categoricas__island_Torgersen', 'categoricas__sex_FEMALE', 'categoricas__sex_MALE', 'numericas__culmen_length_mm', 'numericas__culmen_depth_mm', 'numericas__flipper_length_mm', 'numericas__body_mass_g']
shape X_train preparado: (266, 9)
shape X_test preparado: (67, 9)


## Funcion de evaluacion

Esta funcion entrena un modelo en `train`, predice sobre `test` y retorna su accuracy. Cada modelo se envuelve en un pipeline para que el preprocesamiento se ajuste solo con los datos de entrenamiento.

In [9]:
def build_pipeline(model):
    return Pipeline(
        steps=[
            ("preprocess", make_preprocessor()),
            ("model", model),
        ]
    )

def fit_and_score(model, X_train, X_test, y_train, y_test):
    pipeline = build_pipeline(model)
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    return pipeline, predictions, accuracy


## Mision 2A: prediccion tradicional

En la estrategia tradicional se entrena un unico modelo multiclase para predecir directamente la especie.

In [10]:
traditional_models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "SVC": SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
}

traditional_results = []
traditional_predictions = {}
traditional_pipelines = {}

for name, model in traditional_models.items():
    pipeline, predictions, accuracy = fit_and_score(model, X_train, X_test, y_train, y_test)
    traditional_results.append({"modelo": name, "accuracy_test": accuracy})
    traditional_predictions[name] = predictions
    traditional_pipelines[name] = pipeline

traditional_results = (pd.DataFrame(traditional_results).sort_values("accuracy_test", ascending=False).reset_index(drop=True))

display(traditional_results)

,modelo,accuracy_test
0,Decision Tree,1.000000
1,Random Forest,1.000000
2,SVC,0.985075
3,KNN,0.985075
4,Logistic Regression,0.985075


In [11]:
best_traditional_name = traditional_results.loc[0, "modelo"]
best_traditional_accuracy = traditional_results.loc[0, "accuracy_test"]
best_traditional_predictions = traditional_predictions[best_traditional_name]

print(f"Mejor modelo tradicional: {best_traditional_name}")
print(f"Accuracy en test: {best_traditional_accuracy:.4f}")

Mejor modelo tradicional: Decision Tree
Accuracy en test: 1.0000


In [12]:
cm_traditional = pd.DataFrame(
    confusion_matrix(y_test, best_traditional_predictions),
    index=label_encoder.classes_,
    columns=label_encoder.classes_,
)

display(cm_traditional)
print(classification_report(
    y_test,
    best_traditional_predictions,
    target_names=label_encoder.classes_
))

,Adelie,Chinstrap,Gentoo
Adelie,29,0,0
Chinstrap,0,14,0
Gentoo,0,0,24


              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00        29
   Chinstrap       1.00      1.00      1.00        14
      Gentoo       1.00      1.00      1.00        24

    accuracy                           1.00        67
   macro avg       1.00      1.00      1.00        67
weighted avg       1.00      1.00      1.00        67



## Mision 2B: prediccion jerarquica

Para la estrategia jerarquica tomaremos una decision simple antes de mirar los resultados del test, que es separar primero la especie mas distinguible.

In [13]:
species_means = df.groupby("species")[numeric_features].mean().round(2)
display(species_means)

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g
species,,,,
Adelie,38.82,18.35,191.16,3706.16
Chinstrap,48.83,18.42,195.43,3733.09
Gentoo,47.57,15.00,215.64,5092.44


`Gentoo` se ve claramente separado: tiene mayor masa corporal y aletas mas largas. Por eso usaremos la siguiente jerarquia:

1. Modelo 1: decidir si el pinguino es `Gentoo` o `no Gentoo`.
2. Modelo 2: si no es `Gentoo`, decidir entre `Adelie` y `Chinstrap`.

Para mantener la solucion simple, usaremos arboles de decision en ambos pasos.

In [14]:
gentoo_idx = np.where(label_encoder.classes_ == "Gentoo")[0][0]

y_train_is_gentoo = (y_train == gentoo_idx).astype(int)
y_test_is_gentoo = (y_test == gentoo_idx).astype(int)

print(f"indice de Gentoo: {gentoo_idx}")
print(f"distribucion binaria train, 1 = Gentoo:")
print(pd.Series(y_train_is_gentoo).value_counts().sort_index())

indice de Gentoo: 2
distribucion binaria train, 1 = Gentoo:
0    171
1     95
Name: count, dtype: int64


In [15]:
def hierarchical_predict(model_1, model_2):
    model_1.fit(X_train, y_train_is_gentoo)

    mask_train_not_gentoo = y_train != gentoo_idx # mascara, sirve para filtrar los datos de entrenamiento que no son Gentoo
    # tiene la estructura de [True, False, True, ...]

    model_2.fit(X_train[mask_train_not_gentoo], y_train[mask_train_not_gentoo])

    predictions_model_1 = model_1.predict(X_test)
    final_predictions = []

    for i, predicted_is_gentoo in enumerate(predictions_model_1):
        if predicted_is_gentoo == 1:
            final_predictions.append(gentoo_idx)
        else:
            prediction_second_model = model_2.predict(X_test.iloc[[i]])[0]
            final_predictions.append(prediction_second_model)

    return np.array(final_predictions)


In [16]:
model_1_gentoo = build_pipeline(DecisionTreeClassifier(max_depth=1, random_state=42))
model_2_not_gentoo = build_pipeline(DecisionTreeClassifier(max_depth=1, random_state=42))

Modelo 1: Decision Tree para Gentoo vs resto

Modelo 2: Decision Tree para Adelie vs Chinstrap

In [18]:
hierarchical_predictions = hierarchical_predict(model_1_gentoo, model_2_not_gentoo)
hierarchical_accuracy = accuracy_score(y_test, hierarchical_predictions)

print(f"accuracy en test: {hierarchical_accuracy:.4f}")

accuracy en test: 0.9104


In [19]:
cm_hierarchical = pd.DataFrame(
    confusion_matrix(y_test, hierarchical_predictions),
    index=label_encoder.classes_,
    columns=label_encoder.classes_,
)

display(cm_hierarchical)
print(classification_report(
    y_test,
    hierarchical_predictions,
    target_names=label_encoder.classes_
))

,Adelie,Chinstrap,Gentoo
Adelie,26,1,2
Chinstrap,1,12,1
Gentoo,0,1,23


              precision    recall  f1-score   support

      Adelie       0.96      0.90      0.93        29
   Chinstrap       0.86      0.86      0.86        14
      Gentoo       0.88      0.96      0.92        24

    accuracy                           0.91        67
   macro avg       0.90      0.90      0.90        67
weighted avg       0.91      0.91      0.91        67



In [20]:
comparison = pd.DataFrame({
    "estrategia": ["Tradicional", "Jerarquica"],
    "modelo": [best_traditional_name, "Decision Tree + Decision Tree"],
    "accuracy_test": [best_traditional_accuracy, hierarchical_accuracy],
})

display(comparison)

,estrategia,modelo,accuracy_test
0,Tradicional,Decision Tree,1.000000
1,Jerarquica,Decision Tree + Decision Tree,0.910448


In [21]:
sample_predictions = pd.DataFrame({
    "real": label_encoder.inverse_transform(y_test),
    "prediccion_tradicional": label_encoder.inverse_transform(best_traditional_predictions),
    "prediccion_jerarquica": label_encoder.inverse_transform(hierarchical_predictions),
})

display(sample_predictions.head(12))
display(sample_predictions[sample_predictions["real"] != sample_predictions["prediccion_jerarquica"]])

,real,prediccion_tradicional,prediccion_jerarquica
0,Gentoo,Gentoo,Gentoo
1,Chinstrap,Chinstrap,Chinstrap
2,Adelie,Adelie,Adelie
3,Gentoo,Gentoo,Gentoo
4,Gentoo,Gentoo,Gentoo
5,Gentoo,Gentoo,Gentoo
6,Chinstrap,Chinstrap,Chinstrap
7,Adelie,Adelie,Adelie
8,Adelie,Adelie,Adelie
9,Gentoo,Gentoo,Gentoo


,real,prediccion_tradicional,prediccion_jerarquica
14,Chinstrap,Chinstrap,Adelie
27,Adelie,Adelie,Chinstrap
32,Adelie,Adelie,Gentoo
37,Gentoo,Gentoo,Chinstrap
47,Chinstrap,Chinstrap,Gentoo
65,Adelie,Adelie,Gentoo


## Conclusion

La estrategia tradicional alcanza el mejor rendimiento. La estrategia jerarquica tambien funciona muy bien, pero al usar una eleccion fija y simple de modelos no queda forzada a ser perfecta. 

El costo de equivocarse en la primera decision es alto, porque si un `Gentoo` es clasificado como `no Gentoo`, el segundo modelo nunca lo podra recuperar.

La solucion tradicional es mas directa porque usa un solo modelo. 

La solucion jerarquica es pedagogicamente interesante porque refleja una decision interpretable, primero separar `Gentoo`, que es la especie mas distinguible, y luego clasificar entre `Adelie` y `Chinstrap`.